### Finetune Implementation

## HUẤN LUYỆN MÔ HÌNH LLAMA-3.2-1B VỚI UNSLOTH (QLoRA)
- **Môi trường yêu cầu:** Google Colab (Runtime: T4 GPU) hoặc Máy ảo Linux có GPU NVIDIA.


### BƯỚC 1: Cài đặt thư viện (Chỉ chạy trên Google Colab)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-hyc3v1mf/unsloth_316d1d0a6b0d44a995b5b1f843ac5e7d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-hyc3v1mf/unsloth_316d1d0a6b0d44a995b5b1f843ac5e7d
  Resolved https://github.com/unslothai/unsloth.git to commit 92cee0ff3e4a88cadd5b9ae01b7627bd1c92394a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.6 MB/s eta 0:00:00


### BƯỚC 2: Khai báo thư viện & Đăng nhập Weights & Biases
Sử dụng wandb để theo dõi biểu đồ Loss (sai số) trong quá trình train, giúp phát hiện sớm Overfitting.

In [ ]:
import os

# Optional for more secure: assgin WANDB_KEY with HF_TOKEN in secret key
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_KEY", "PASTE_API_KEY_HERE")
os.environ["WANDB_SILENT"] = "true"

import torch
import wandb
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template
from transformers import TrainingArguments

# Read and import WANDB_API_KEY
wandb.login()

True

### BƯỚC 3: Nạp Mô hình gốc (Base Model) & Tokenizer
Sử dụng Unsloth để nạp mô hình Llama-3.2-1B ở định dạng 4-bit, giúp tiết kiệm tối đa VRAM.

In [ ]:
# --- NẠP MÔ HÌNH VÀ CẤU HÌNH TOKENIZER CHUẨN ---
max_seq_length = 8192 # # Nâng lên 8192 để đọc context RAG thoải mái
dtype = None # Unsloth tự động chọn (Bfloat16 cho Ampere+, Float16 cho T4/V100)
load_in_4bit = True # Bắt buộc True để dùng QLoRA (Tiết kiệm VRAM)


print("Đang nạp mô hình Base Llama-3.2-1B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct", # BẮT BUỘC DÙNG MODEL GỐC NÀY
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Cấu hình Tokenizer nhận diện định dạng Llama-3 chuẩn
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Llama 3.2 dùng chung template với 3.1
)
print("Nạp mô hình thành công!")


Đang nạp mô hình Base Llama-3.2-1B-Instruct...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Nạp mô hình thành công!


#### Kiểm tra token, ép token end và token padding phải thống nhất

### BƯỚC 4: Cấu hình QLoRA Adapters
- Không train toàn bộ 1 tỷ tham số, mà chỉ thêm vào các ma trận nhỏ (adapters).
- Rank (r) = 32 giúp mô hình học các khái niệm y khoa phức tạp.

In [ ]:
# --- CẤU HÌNH QLORA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Mức độ phức tạp của LoRA (Khuyên dùng: 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # Áp dụng lên tất cả Attention & MLP
    lora_alpha = 64, # Thường gấp đôi r
    lora_dropout = 0, # Tối ưu hóa (Unsloth khuyên dùng 0)
    bias = "none",    # Tối ưu hóa
    use_gradient_checkpointing = "unsloth", # Cực kỳ quan trọng: Giảm 30% VRAM
    random_state = 42,
    use_rslora = False,
)

### BƯỚC 5: Nạp Tập dữ liệu Huấn luyện
Đọc file `final_augmented_train.jsonl` (đã được format sẵn chuẩn ChatML ở Giai đoạn 2).


In [ ]:
# Đảm bảo bạn đã upload train_ready.jsonl và eval_ready.jsonl lên Colab
train_dataset = load_dataset("json", data_files="/content/train_ready.jsonl", split="train")
eval_dataset = load_dataset("json", data_files="/content/eval_ready.jsonl", split="train")

# Hàm chuyển đổi mảng 'messages' thành chuỗi ChatML chuẩn
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

# Áp dụng cho cả 2 tập dữ liệu
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

print(f"Số mẫu Train: {len(train_dataset)} | Số mẫu Eval: {len(eval_dataset)}")

Số mẫu Train: 2830 | Số mẫu Eval: 360


### BƯỚC 6: Thiết lập tham số Huấn luyện (SFTTrainer)
Với dữ liệu chất lượng cao, chỉ cần 1 đến 2 epochs là đủ. Train nhiều hơn dễ gây ảo giác (overfit).


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset, # Đưa tập Eval vào để chấm điểm
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 3, # Tăng lên 3 epochs để mô hình ngấm sâu kiến thức
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        eval_strategy = "steps", # Đánh giá sau mỗi X bước
        eval_steps = 50, # Cứ 50 bước train thì lấy tập eval ra test
        save_strategy = "steps",
        save_steps = 50, # Lưu checkpoint cùng lúc với eval
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "wandb",
        load_best_model_at_end = True, # Giữ lại checkpoint có Validation Loss thấp nhất
    ),
)

### BƯỚC 7: Bắt đầu Huấn luyện 🚀
Theo dõi cột `Loss`, nếu nó giảm dần dần từ (ví dụ) 2.0 xuống 0.7 - 0.5 là mô hình đang học rất tốt.

In [ ]:
# --- BẮT ĐẦU HUẤN LUYỆN ---
print("Bắt đầu huấn luyện...")

# Initialize wandb for logging
import wandb
wandb.init(project="llama3-2-1b-pet-chatbot")

trainer_stats = trainer.train()

# End the wandb run after training
wandb.finish()

Bắt đầu huấn luyện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,830 | Num Epochs = 3 | Total steps = 1,062
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Step,Training Loss,Validation Loss
50,1.405973,1.386748
100,0.987642,1.327274
150,0.713501,1.419925
200,0.496123,1.607039
250,0.231558,1.700423
300,0.139780,1.851868
350,0.104408,1.849605
400,0.065257,1.964777
450,0.048830,2.007674
500,0.046676,2.076914


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

### BƯỚC 8: Kiểm thử trực tiếp Mô hình sau khi Train (Inference Test)
Hãy hỏi thử một câu RAG để xem mô hình trả lời thế nào.

In [ ]:
import warnings
import transformers

# KHÓA TẤT CẢ CẢNH BÁO (WARNINGS)
# Ẩn cảnh báo FutureWarning và UserWarning của Python
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

FastLanguageModel.for_inference(model)

system_prompt = "Bạn là trợ lý AI chuyên gia thú y tận tâm. Hãy trả lời câu hỏi dựa trên thông tin tham khảo được cung cấp một cách chính xác và đồng cảm. Tuyệt đối không tự bịa đặt."
context = "Bệnh Care ở chó (Canine Distemper) là bệnh truyền nhiễm nguy hiểm do virus gây ra, ảnh hưởng đến hệ hô hấp, tiêu hóa và thần kinh. Chó chưa tiêm phòng rất dễ mắc bệnh."
question = "Chó nhà tôi chưa tiêm phòng, dạo này thấy ho và tiêu chảy. Liệu có phải bệnh Care không?"

# Sử dụng messages list thay vì ghép chuỗi thủ công
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI:\n{question}"}
]

# Tự động render prompt chuẩn
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Kích hoạt bot trả lời
    return_tensors = "pt",
).to("cuda")

# Sinh câu trả lời
outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
# Chỉ in ra phần token mới sinh ra
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n--- CÂU TRẢ LỜI CỦA MÔ HÌNH ---")
print(response.strip())


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- CÂU TRẢ LỜI CỦA MÔ HÌNH ---
Chào bạn, có thể có một số nguyên nhân khác như virus parvovirus, virus mèo hoặc virus ký sinh trùng. Bạn nên đưa chó đến bác sĩ thú y để kiểm tra và điều trị sớm.


### BƯỚC 9: Xuất Mô hình (GGUF Export)
- Định dạng GGUF (chạy bằng llama.cpp hoặc Ollama) là đỉnh cao của sự tối ưu.
- Nó nén file chỉ còn khoảng ~1GB và có thể chạy cực mượt trên CPU hoặc VPS giá rẻ ở Giai đoạn 4.

In [ ]:
# Đọc dữ liệu từ mounted google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- XUẤT MÔ HÌNH (GGUF) ---
print("Huấn luyện xong, đang xuất file GGUF...")
# Xuất định dạng lượng hóa 4-bit q4_k_m cực kỳ tối ưu cho CPU và VPS nhẹ
model.save_pretrained_gguf("Llama-3.2-1B-Pet-Chatbot-GGUF", tokenizer, quantization_method = "q4_k_m")
print("Xuất file hoàn tất!")

Huấn luyện xong, đang xuất file GGUF...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:33<00:00, 33.70s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:48<00:00, 48.21s/it]


Unsloth: Merge process complete. Saved to `/content/Llama-3.2-1B-Pet-Chatbot-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Llama-3.2-1B-Pet-Chatbot-GGUF_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Llama-3.2-1B-Pet-Chatbot-GGUF_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model Llama-3.2-1B-Pet-Chatbot-GGUF_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to Llama-3.2-1B-Pet-Chatbot-GGUF_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f Llama-3.2-1B-Pet-Chatbot-GGUF_gguf/Modelfile
Xuất file hoàn tất!


In [ ]:
import shutil
import os

# Đường dẫn thư mục gốc chứa model vừa xuất
source_dir = "/content/Llama-3.2-1B-Pet-Chatbot-GGUF"

# Đường dẫn đích trên Google Drive của bạn (có thể đổi tên thư mục Pet_Chatbot_Model tùy ý)
drive_dir = "/content/drive/MyDrive/Pet_Chatbot_Model"

print(f"Đang copy model từ {source_dir} lên Google Drive ({drive_dir})...")
shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
print("✅ Đã sao lưu Model lên Google Drive an toàn!")

Đang copy model từ /content/Llama-3.2-1B-Pet-Chatbot-GGUF lên Google Drive (/content/drive/MyDrive/Pet_Chatbot_Model)...
✅ Đã sao lưu Model lên Google Drive an toàn!
